In [4]:
import time
import numpy as np
import networkx as nx
import cvxpy as cp

# =========================
# 1. Comprehensive Graph Generator
# =========================
def generate_graphs(num_graphs=1, graph_type='erdos_renyi', n=50, seed_start=0, **kwargs):
    """
    Generates a list of graphs based on the specified topology.
    
    Supported types and their kwargs:
    - 'erdos_renyi': p (edge probability, default 0.3)
    - 'regular': d (degree, default 3) - Note: n * d must be even.
    - 'barabasi_albert': m (edges to attach from a new node, default 3)
    - 'complete': no extra kwargs
    """
    graphs = []
    for i in range(num_graphs):
        current_seed = seed_start + i
        
        if graph_type == 'erdos_renyi':
            p = kwargs.get('p', 0.3)
            G = nx.erdos_renyi_graph(n, p, seed=current_seed)
        
        elif graph_type == 'regular':
            d = kwargs.get('d', 3)
            G = nx.random_regular_graph(d, n, seed=current_seed)
            
        elif graph_type == 'barabasi_albert':
            m = kwargs.get('m', 3)
            G = nx.barabasi_albert_graph(n, m, seed=current_seed)
            
        elif graph_type == 'complete':
            G = nx.complete_graph(n)
            
        else:
            raise ValueError(f"Unsupported graph_type: '{graph_type}'.")

        # Name the graph for tracking in the pipeline
        G.name = f"{graph_type.upper()}_n{n}_i{i}"
        graphs.append(G)
        
    return graphs

# =========================
# 2. Solve SDP relaxation
# =========================
def solve_sdp(G):
    n = G.number_of_nodes()
    X = cp.Variable((n, n), symmetric=True)
    constraints = [X >> 0, cp.diag(X) == 1]

    # Graph Laplacian formulation (Fastest for CVXPY)
    L = nx.laplacian_matrix(G).toarray()
    obj = cp.trace(L @ X) / 4

    prob = cp.Problem(cp.Maximize(obj), constraints)
    prob.solve(solver=cp.SCS)

    return X.value, prob.value

# =========================
# 3. Factor SDP solution
# =========================
def factorize_sdp(X):
    eigvals, eigvecs = np.linalg.eigh(X)
    eigvals = np.clip(eigvals, 0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals))

# =========================
# 4. Hyperplane rounding
# =========================
def gw_rounding(G, V, num_samples=1000):
    n, d = V.shape
    edges = np.array(list(G.edges()))

    # Fully vectorized random hyperplanes
    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)

    signs = np.sign(V @ R)
    signs[signs == 0] = 1 

    cuts = np.sum(signs[edges[:, 0]] != signs[edges[:, 1]], axis=0)

    best_idx = np.argmax(cuts)
    return cuts[best_idx], signs[:, best_idx]

# =========================
# 5. Full Pipeline with Timing & Metrics
# =========================
def goemans_williamson(G, num_samples=1000):
    # Time the SDP solver
    t0 = time.perf_counter()
    X, sdp_val = solve_sdp(G)
    sdp_time = time.perf_counter() - t0

    # Time the Rounding phase
    t1 = time.perf_counter()
    V = factorize_sdp(X)
    cut, partition = gw_rounding(G, V, num_samples)
    rounding_time = time.perf_counter() - t1

    approx_ratio = cut / sdp_val if sdp_val > 0 else 0

    return {
        "graph_name": getattr(G, "name", "Unnamed"),
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "sdp_val": sdp_val,
        "cut_val": cut,
        "approx_ratio": approx_ratio,
        "sdp_time": sdp_time,
        "rounding_time": rounding_time,
        "total_time": sdp_time + rounding_time
    }

# =========================
# RUN BATCH EXPERIMENTS
# =========================
if __name__ == "__main__":
    
    # 1. Generate a batch of regular graphs (e.g., 3 graphs, 50 nodes, degree 4)
    print("Generating graphs...")
    graphs = generate_graphs(
        num_graphs=3, 
        graph_type='regular', 
        n=50, 
        d=4, 
        seed_start=42
    )
    
    # 2. Process them and collect results
    all_results = []
    
    print("\nStarting Goemans-Williamson MaxCut Pipeline...\n" + "-"*55)
    for G in graphs:
        res = goemans_williamson(G, num_samples=2000)
        all_results.append(res)
        
        # Print summary for the current graph
        print(f"Graph: {res['graph_name']} (|V|={res['nodes']}, |E|={res['edges']})")
        print(f"  ├─ SDP Upper Bound: {res['sdp_val']:.2f}")
        print(f"  ├─ Actual Cut:      {res['cut_val']}")
        print(f"  ├─ Approx Ratio:    {res['approx_ratio']:.4f}  <-- (Optimality bound)")
        print(f"  └─ Time Taken:      {res['total_time']:.3f}s (SDP: {res['sdp_time']:.3f}s, Rounding: {res['rounding_time']:.3f}s)\n")

    # 3. Aggregate metrics across the batch
    avg_ratio = np.mean([r['approx_ratio'] for r in all_results])
    avg_time = np.mean([r['total_time'] for r in all_results])
    
    print("-" * 55)
    print(f"BATCH SUMMARY:")
    print(f"Average Approx Ratio: {avg_ratio:.4f}")
    print(f"Average Time / Graph: {avg_time:.3f}s")

Generating graphs...

Starting Goemans-Williamson MaxCut Pipeline...
-------------------------------------------------------
Graph: REGULAR_n50_i0 (|V|=50, |E|=100)
  ├─ SDP Upper Bound: 87.72
  ├─ Actual Cut:      84
  ├─ Approx Ratio:    0.9576  <-- (Optimality bound)
  └─ Time Taken:      0.045s (SDP: 0.041s, Rounding: 0.004s)

Graph: REGULAR_n50_i1 (|V|=50, |E|=100)
  ├─ SDP Upper Bound: 87.66
  ├─ Actual Cut:      84
  ├─ Approx Ratio:    0.9582  <-- (Optimality bound)
  └─ Time Taken:      0.045s (SDP: 0.042s, Rounding: 0.003s)

Graph: REGULAR_n50_i2 (|V|=50, |E|=100)
  ├─ SDP Upper Bound: 86.29
  ├─ Actual Cut:      82
  ├─ Approx Ratio:    0.9502  <-- (Optimality bound)
  └─ Time Taken:      0.076s (SDP: 0.073s, Rounding: 0.004s)

-------------------------------------------------------
BATCH SUMMARY:
Average Approx Ratio: 0.9554
Average Time / Graph: 0.056s
